# Vision Transformer for CIFAR-10 Image Classification

This notebook supports the assignment report. It explains the workflow and loads saved metrics/plots when available, so it does not require full training every time.

## Objective

Implement a Vision Transformer from scratch for CIFAR-10 and compare it with a Hybrid CNN + MLP model and a pretrained ResNet18 transfer learning model.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import torch
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
DATA_ROOT = PROJECT_ROOT
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
print('Project root:', PROJECT_ROOT)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## Dataset Loading

The dataset is loaded locally from `D:\\cifar-10-python` with `download=False`. The training split is divided into 45,000 training images and 5,000 validation images using a fixed seed. The test split remains 10,000 images.

In [ ]:
from src.config import CIFAR10_CLASSES
from src.data import validate_cifar10_root, get_cifar10_dataloaders

validate_cifar10_root(DATA_ROOT)
loaders = get_cifar10_dataloaders(DATA_ROOT, batch_size=128, smoke_test=True)
print('Classes:', CIFAR10_CLASSES)
print('Smoke train batches:', len(loaders['train']))
print('Smoke val batches:', len(loaders['val']))
print('Smoke test batches:', len(loaders['test']))


## Sample Images and Augmentation

The project saves original CIFAR-10 samples and augmented examples to `outputs/plots/` during training.

In [ ]:
for image_path in [OUTPUT_DIR / 'plots' / 'cifar10_samples.png', OUTPUT_DIR / 'plots' / 'cifar10_augmentations.png']:
    if image_path.exists():
        display(Image(filename=str(image_path)))
    else:
        print('Missing plot:', image_path)


## Preprocessing

ViT and Hybrid models use random crop, random horizontal flip, tensor conversion, and CIFAR-10 normalization for training. Validation and test transforms omit random augmentation. ResNet18 uses 224x224 resized inputs and ImageNet normalization because its pretrained weights come from ImageNet.

## Vision Transformer from Scratch

The ViT divides each 32x32 image into non-overlapping patches. With patch size 4, the image becomes an 8x8 grid of 64 patches. A Conv2d patch embedding layer projects each patch to a token vector. A learnable class token is prepended, learnable positional embeddings are added, and the sequence is passed through Transformer encoder blocks. The final class-token representation is used for classification.

In [ ]:
from src.models.vit import VisionTransformer, count_vit_patches

vit = VisionTransformer()
print(vit)
print('Default number of patches:', count_vit_patches(32, 4))


## Hybrid CNN + MLP

The Hybrid CNN + MLP baseline learns local image features using convolutional blocks, pooling, and batch normalization, then classifies the pooled features with an MLP head. It does not use self-attention or Transformer blocks.

In [ ]:
from src.models.hybrid_cnn_mlp import HybridCNNMLP

hybrid = HybridCNNMLP()
print(hybrid)


## ResNet18 Transfer Learning

The ResNet18 model uses ImageNet-pretrained weights through `torchvision.models.resnet18(weights="IMAGENET1K_V1")`. The backbone is frozen at first and the final classifier is replaced for CIFAR-10. The script can optionally fine-tune `layer4`.

## Commands

Smoke test:

```bat
conda activate vit-cifar10
cd /d D:\cifar-10-python
python train_all.py --smoke-test
```

Fast ViT tuning:

```bat
python train_all.py --tune-vit-fast --smoke-test
```

Full training:

```bat
python train_all.py
```

Evaluation:

```bat
python evaluate_all.py
```

## Metrics Summary

The final comparison table is saved to `outputs/metrics/metrics_summary.csv` and `outputs/metrics/metrics_summary.json`.

In [ ]:
metrics_csv = OUTPUT_DIR / 'metrics' / 'metrics_summary.csv'
if metrics_csv.exists():
    metrics_df = pd.read_csv(metrics_csv)
    display(metrics_df)
else:
    print('Metrics summary not found yet. Run train_all.py or evaluate_all.py first.')


## Training Curves and Confusion Matrices

In [ ]:
for model_name in ['vit', 'hybrid', 'resnet']:
    for suffix in ['accuracy_curve', 'loss_curve', 'confusion_matrix']:
        image_path = OUTPUT_DIR / 'plots' / f'{model_name}_{suffix}.png'
        if image_path.exists():
            print(image_path.name)
            display(Image(filename=str(image_path)))


## Correct and Incorrect Predictions

In [ ]:
for model_name in ['vit', 'hybrid', 'resnet']:
    for suffix in ['correct_predictions', 'incorrect_predictions']:
        image_path = OUTPUT_DIR / 'plots' / f'{model_name}_{suffix}.png'
        if image_path.exists():
            print(image_path.name)
            display(Image(filename=str(image_path)))


## Hyperparameter Tuning Results

In [ ]:
tuning_csv = OUTPUT_DIR / 'metrics' / 'vit_hyperparameter_tuning.csv'
if tuning_csv.exists():
    display(pd.read_csv(tuning_csv))
else:
    print('No tuning CSV found yet. Run: python train_all.py --tune-vit-fast --smoke-test')


## Discussion

Compare the models by accuracy, F1-score, training time, memory use, and inference speed. The ViT demonstrates self-attention from scratch, the Hybrid CNN + MLP provides a compact convolutional baseline, and ResNet18 shows the effect of transfer learning from a large-scale dataset.

Common CIFAR-10 confusions include cat/dog, deer/horse, and automobile/truck. Use the incorrect prediction grids and confusion matrices to support the final report discussion.

## Conclusion

This notebook and the saved outputs provide an assignment-friendly record of implementation, preprocessing, training, tuning, evaluation, and local deployment results.